In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from datetime import datetime
import glob
from show import *
from utils import *


In [2]:
def remap(data, header, nx=720, ny=1800, **kwargs):
    from transforms import ToSpherical
    from scipy.ndimage import map_coordinates

    view = View.from_header(header).update(**kwargs)
    transform = ~ToSpherical() - view.to_carrington(**kwargs)

    lat_spread = 2
    dlon = 360 / ny
    dlat = lat_spread / (nx - 1)


    grid = np.mgrid[-lat_spread / 2:lat_spread / 2 + dlat / 2:dlat, :360:dlon]
    grid[0] = np.arcsin(grid[0].clip(-1,1)) * 180 / np.pi

    grid, alpha = transform(grid)
    data_ = map_coordinates(np.nan_to_num(data), grid, cval=np.nan)

    return data_, grid, alpha


In [98]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/2026/blos/*')))
dates = np.array([datetime.fromisoformat(file.split('/')[-1].split('_')[-3]) for file in files])

#t = np.where(np.all([dates > datetime(2026,4,9),
#                     dates < datetime(2026,5,1)], axis=0))[0] # CR2310

t = np.where(np.all([dates > datetime(2026,5,1),
                     dates < datetime(2026,5,28)], axis=0))[0] # CR2311

#t = np.where(np.all([dates > datetime(2026,5,28),
#                     dates < datetime(2026,6,24)], axis=0))[0] # CR2312

#t = np.where(np.all([dates > datetime(2026,6,24),
#                     dates < datetime(2026,7,22)], axis=0))[0] # CR2313

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,8,31)], axis=0))[0] # CR2314

#t = np.where(np.all([dates > datetime(2026,8,31),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2315

#t = np.where(np.all([dates > datetime(2026,7,8),
#                     dates < datetime(2026,8,8)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,9,14),
#                     dates < datetime(2026,10,13)], axis=0))[0]

#t = np.where(np.all([dates > datetime(2026,7,22),
#                     dates < datetime(2026,9,29)], axis=0))[0] # CR2314 + CR2315

files = files[t]
files[0], files[-1], len(files)

(np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260501T020003_V202610192232C_0645010501.fits'),
 np.str_('/home/ulyanov/data/solo/phi/2026/blos/phi-fdt-blos_20260527T230009_V202610192358C_0645270508.fits'),
 206)

In [111]:
N = 0

LL = 0
GG = 0
LG = 0

LF = 0
GF = 0

P = [-0.85000247,  5.93711722, -1.05000067, -1.53541529, -0.13794807, -0.11734419]

for file in files[:]:

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
    data_, grid, alpha = remap(data, header, delta_t=20, correct_dr=True, minmu=0.1)
    xc, yc, rsun = header['CRPIX2'] - 1, header['CRPIX1'] - 1, header['RSUN_ARC'] / header['CDELT1']

    x, y = grid
    x, y = (x + x0 - 1024) / 1024, (y + y0 - 1024) / 1024
    #x, y = (x + x0 - xc) / rsun, (y + y0 - yc) / rsun
    z = 1 / alpha
    n = (~np.isnan(data_)).astype(float)

    #data_ -= P[0] * x ** 2 + P[1] * x * y + P[2] * y ** 2 + P[3] * x + P[4] * y + P[5]


    l = np.expand_dims(z, (-2,-1))
    #g = np.moveaxis(np.array([[x ** 3, x ** 2 * y, x * y ** 2, y ** 3, x ** 2, x * y, y ** 2, x, y, n]]), (0,1), (-2,-1))
    g = np.moveaxis(np.array([[x ** 2, x * y, y ** 2, x, y, n]]), (0,1), (-2,-1))
    #g = np.moveaxis(np.array([[x, y, n]]), (0,1), (-2,-1))
    f = np.expand_dims(data_, (-2,-1))

    ll = np.swapaxes(l, -2, -1) @ l
    gg = np.swapaxes(g, -2, -1) @ g
    lg = np.swapaxes(l, -2, -1) @ g

    lf = np.swapaxes(l, -2, -1) @ f
    gf = np.swapaxes(g, -2, -1) @ f


    N += np.nan_to_num(n)
    w = np.expand_dims(n / N, (-2, -1))

    LL += np.nan_to_num((ll - LL) * w)
    GG += np.nan_to_num((gg - GG) * w)
    LG += np.nan_to_num((lg - LG) * w)
    LF += np.nan_to_num((lf - LF) * w)
    GF += np.nan_to_num((gf - GF) * w)


LL[N == 0] = np.nan
GG[N == 0] = np.nan
LG[N == 0] = np.nan

LF[N == 0] = np.nan
GF[N == 0] = np.nan

N[N == 0] = np.nan

/tmp/ipykernel_31701/793498659.py:46: RuntimeWarning: invalid value encountered in divide
  w = np.expand_dims(n / N, (-2, -1))


In [114]:
W = np.expand_dims(N ** 2, (-2,-1)) #* LL


W[200:-200] = np.nan
#W[:-200] = np.nan

LLi = np.linalg.inv(LL)

Q = LLi @ LF
R = LLi @ LG

GL = np.swapaxes(LG, -2, -1)
D = np.linalg.inv(np.nanmean((GG - GL @ R) * W, axis=(0,1)))
V = np.nanmean((GF - GL @ Q) * W, axis=(0,1))
P = D @ V

Q_ = Q - R @ P

P = np.squeeze(P)
Q = np.squeeze(Q)
Q_ = np.squeeze(Q_)

print(P)

[-14.66598199  12.0077255  -11.59086046  -2.77144792   0.11041249
   1.10190585]


In [80]:
xi, yi = np.mgrid[:2048,:2048]
xi, yi = xi / 1024 - 1, yi / 1024 - 1

#bias = P[0] * xi ** 2 + P[1] * xi * yi + P[2] * yi ** 2 + P[3] * xi + P[4] * yi + P[5]
bias = P[0] * xi + P[1] * yi + P[2]

In [37]:
plt.figure(figsize=(10,10))
plt.imshow(bias, cmap='seismic', vmin=-10, vmax=10)
plt.tight_layout()

In [63]:
plt.figure(figsize=(12,6))
plt.imshow(N, 'gray', origin='lower', aspect='auto')
plt.tight_layout()

In [81]:
show_map(Q_, vmin=-500, vmax=500, sine_lat=True)

In [115]:
nx = len(Q)

lat = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

#flux = np.nanmean(Q * N ** 2, axis=1) / np.nanmean(N ** 2, axis=1)
#flux_ = np.nanmean(Q_ * N ** 2, axis=1) / np.nanmean(N ** 2, axis=1)

flux = np.nanmean(Q, axis=1)
flux_ = np.nanmean(Q_, axis=1)

plt.figure(figsize=(10,8))
plt.plot(lat, flux)
plt.plot(lat, flux_)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

/tmp/ipykernel_31701/941732730.py:8: RuntimeWarning: Mean of empty slice
  flux = np.nanmean(Q, axis=1)
/tmp/ipykernel_31701/941732730.py:9: RuntimeWarning: Mean of empty slice
  flux_ = np.nanmean(Q_, axis=1)


In [116]:
file = '/home/ulyanov/data/sdo/hmi/synoptic_maps/hmi.Synoptic_Mr_720s.2311.synopMr.fits'

with fits.open(file) as hdul:
    header = hdul[0].header.copy()
    data_ = hdul[0].data.copy()

data_ = rebin(data_, 2)
#data_ = gaussian_filter(data_, 2, truncate=3)
header['T_OBS']

'2026.05.26_12:31:00_TAI'

In [117]:
show_map(data_, vmin=-500, vmax=500, sine_lat=True)

In [118]:
#flux_hmi = np.nanmean(data_ * Z2, axis=1) / np.nanmean(Z2, axis=1)
flux_hmi = np.nanmean(data_, axis=1)

nx, ny = data_.shape
lat_hmi = np.arcsin(np.arange(nx) / (nx - 1) * 2 - 1) * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(lat_hmi, flux_hmi, label='HMI')
plt.plot(lat, flux, label='FDT')
plt.plot(lat, flux_, label='FDT (corrected)', color='red')

plt.xlabel('Latitude, degrees')
plt.ylabel(r'Flux, Mx/cm$^2$')
plt.xlim(-90,90)

plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.legend()
plt.tight_layout()

/tmp/ipykernel_31701/4225042231.py:2: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_, axis=1)


In [16]:
np.nanmean(Q_)

np.float64(-0.030602559551932078)

In [28]:
files[127]

np.str_('/home/ulyanov/data/solo/phi/2026/blos_/phi-fdt-blos_20260830T225003_V202610202050C_0648300501.fits')

In [31]:
from processing import clone_fits

clone_fits(files[127], 'phi-fdt-map_20260830T225003.fits', Q_ - np.nanmean(Q_))